# 04_张量与自动求导

**学习目标：** 理解张量的shape/dtype/device，并通过小计算解释梯度。

适合：读过CS50P但基础不熟、刚开始深度学习。按顺序学习；每课先看讲解、跑示例、做练习，再查答案。


## 本课要点

Tensor像NumPy数组，但可以放在GPU并跟踪求导。参数调整的方向由梯度提供，学习率决定一次走多远。这里只需要理解平方函数的斜率：f(w)=(w-3)^2的导数是2(w-3)。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## NumPy到张量，再加batch维


In [ ]:
import torch
torch.set_num_threads(4)
clip, label, _ = L.load_clip()
x = torch.from_numpy(clip).unsqueeze(0)
y = torch.tensor([label], dtype=torch.long)
print(x.shape, x.dtype, x.device, '标签dtype', y.dtype)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = x.to(device)
print('移动后：', x.device)


## 手算与自动求导一致


In [ ]:
w = torch.tensor(1.0, requires_grad=True)
loss = (w - 3).square()
loss.backward()
print('loss', loss.item(), 'gradient', w.grad.item())
assert w.grad.item() == -4
with torch.no_grad():
    w -= 0.1 * w.grad
print('更新后参数', w.item(), '更接近目标3')


## 梯度会累积


In [ ]:
w = torch.tensor(1.0, requires_grad=True)
(w - 3).square().backward()
(w - 3).square().backward()
print('两次backward后', w.grad.item())
w.grad = None
(w - 3).square().backward()
print('清空后一次backward', w.grad.item())


## 动手练习

设w=5，计算(w-3)^2的梯度。将梯度数值填入your_answer，并解释更新方向。

显示“待完成”表示练习尚未作答；代码示例运行成功不等于练习完成。


In [ ]:
your_answer = None  # TODO：先手算，再用torch验证
if your_answer is None:
    print('待完成：导数是2*(w-3)')
else:
    assert your_answer == 4
    print('练习通过；正梯度意味着梯度下降会把w减小')


## 常见误区

标签一般是long，模型输入一般是float32；CPU和GPU张量不能直接混算。


## 过关检查与选读

解释requires_grad、backward、zero_grad；区分loss的值与参数梯度。

选做：改变示例中一个参数，先预测现象再运行，并记下原因。

[对应参考答案](../参考答案/04_张量与自动求导.ipynb) · [官方选读](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

完成后在README.md#学习记录写下一句自己的解释。
